# 🦅 AerialViews+ : Autonomous 4K Ambient Video Curation Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/naveeneppalapally/AerialViews-Plus/blob/main/notebooks/curate_10000_videos.ipynb)

This cloud pipeline discovers, audits, and verifies up to **10,000 pristine 4K ambient video streams** for **AerialViews+** (Android TV screensaver).

### 🛡️ What This Pipeline Purges:
1. **Talking Heads & Vloggers:** Human faces, walking tours, travel vlog commentary.
2. **Text & Watermarks:** Title banners, channel logos, "SUBSCRIBE" popups, timestamps, stock tags.
3. **Fake CGI & Particle Loops:** Still JPEG photos with artificial digital rain/snow overlays.
4. **Shaky Handheld Cameras:** Walking/biking jitter, fisheye/barrel distortion.
5. **Commercial Clutter:** Storefronts, resort ads, price tags.

### ⚡ Powered By:
* **Sub-Second Stream Byte Seeking:** Decodes candidate video frames directly from the inside without downloading full files.
* **GPU Motion Dynamics:** Dense Gunnar Farnebäck Optical Flow, static background SSIM, and angular jerk with **Fluid / Ocean Surf Protection**.
* **OpenCLIP GPU Filter:** Evaluates 2,200 fps in FP16 autocast across an 8-category ambient subspace.
* **Gemini Vision Gold Audit:** Multi-frame multimodal inspection with native micro-watermark OCR.
* **Persistent Google Drive WAL:** Zero data loss across Colab session restarts.



## Step 1: Install Dependencies & Verify GPU Acceleration


In [ ]:
# Install required cloud packages
!pip install -q yt-dlp open-clip-torch google-genai opencv-python-headless pillow

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("WARNING: Running on CPU. For best performance, go to Runtime -> Change runtime type -> T4 or L4/A100 GPU.")



## Step 2: Connect Google Drive & Set Gemini API Key
* **Google Drive:** Mounts your Drive to save progress in `AerialViews_Curator/state/` so you never lose audited candidates.
* **Gemini API Key:** Add `GEMINI_API_KEY` to Colab Secrets (the 🔑 Key icon on the left sidebar), or enter it directly below.



In [ ]:
import os
import getpass
from google.colab import drive

# 1. Mount Google Drive for persistent state logging
try:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AerialViews_Curator'
except Exception as e:
    print("Drive mount skipped or failed; using local Colab storage.")
    BASE_DIR = '/content/AerialViews_Curator'

os.makedirs(os.path.join(BASE_DIR, 'state'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'output'), exist_ok=True)
print(f"Working Directory: {BASE_DIR}")

# 2. Acquire Gemini API Key
GEMINI_API_KEY = ""
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Enter your Gemini API Key: ").strip()

print(f"Gemini API Key configured: {'YES (ends in ...' + GEMINI_API_KEY[-4:] + ')' if GEMINI_API_KEY else 'NO'}")



## Step 3: Initialize the 5-Stage Verification Engines


In [ ]:
import io
import json
import re
import time
import gzip
import subprocess
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Set
import numpy as np
import cv2
from PIL import Image
import yt_dlp
import torch
import open_clip
from google import genai
from google.genai import types

# -------------------------------------------------------------
# Module 1: Sub-Second Stream Byte Seeker (Header-Forwarded)
# -------------------------------------------------------------
class StreamByteSeeker:
    def __init__(self, target_height: int = 360, timeout_sec: int = 15):
        self.target_height = target_height
        self.timeout_sec = timeout_sec
        self.ydl_opts = {
            "quiet": True,
            "no_warnings": True,
            "noplaylist": True,
            "extractor_args": {
                "youtube": {
                    "player_client": ["android", "ios", "web"]
                }
            }
        }

    def resolve_stream(self, video_id: str) -> Optional[Tuple[str, str]]:
        url = f"https://www.youtube.com/watch?v={video_id}"
        try:
            with yt_dlp.YoutubeDL(self.ydl_opts) as ydl:
                info = ydl.extract_info(url, download=False)
                formats = info.get("formats", [])
                
                # Priority 1: Exact target resolution
                for f in formats:
                    if f.get("vcodec") != "none" and f.get("url") and "manifest" not in f.get("url"):
                        if f.get("height") == self.target_height:
                            ua = f.get("http_headers", {}).get("User-Agent", "Mozilla/5.0")
                            return f["url"], ua
                
                # Priority 2: 240p or 480p fallback
                for f in formats:
                    if f.get("vcodec") != "none" and f.get("url") and "manifest" not in f.get("url"):
                        if f.get("height") in [240, 480]:
                            ua = f.get("http_headers", {}).get("User-Agent", "Mozilla/5.0")
                            return f["url"], ua

                # Priority 3: First available playable video
                for f in formats:
                    if f.get("vcodec") != "none" and f.get("url") and "manifest" not in f.get("url"):
                        ua = f.get("http_headers", {}).get("User-Agent", "Mozilla/5.0")
                        return f["url"], ua
        except Exception:
            return None
        return None

    def extract_burst(
        self, 
        stream_url: str, 
        user_agent: str, 
        timestamp_sec: float, 
        duration_sec: float = 1.0, 
        fps: int = 4
    ) -> List[Image.Image]:
        cmd = [
            "ffmpeg", "-y",
            "-headers", f"User-Agent: {user_agent}\r\n",
            "-ss", f"{timestamp_sec:.2f}",
            "-reconnect", "1",
            "-reconnect_streamed", "1",
            "-reconnect_delay_max", "2",
            "-fflags", "+nobuffer+fastseek+discardcorrupt",
            "-i", stream_url,
            "-an", "-sn", "-dn",
            "-t", f"{duration_sec:.2f}",
            "-vf", f"fps={fps}",
            "-f", "image2pipe",
            "-vcodec", "mjpeg",
            "pipe:1"
        ]
        
        try:
            res = subprocess.run(cmd, capture_output=True, timeout=self.timeout_sec)
            if res.returncode != 0 or len(res.stdout) < 1000:
                return []
            
            data = res.stdout
            images = []
            curr = 0
            while curr < len(data):
                start = data.find(b"\xff\xd8", curr)
                if start == -1:
                    break
                end = data.find(b"\xff\xd9", start + 2)
                if end == -1:
                    break
                jpeg_bytes = data[start:end+2]
                try:
                    img = Image.open(io.BytesIO(jpeg_bytes)).convert("RGB")
                    images.append(img)
                except Exception:
                    pass
                curr = end + 2
                
            return images
        except Exception:
            return []

# -------------------------------------------------------------
# Module 2: Motion Dynamics Engine (Fluid & Ocean Protected)
# -------------------------------------------------------------
def compute_masked_ssim(img1: np.ndarray, img2: np.ndarray, mask: np.ndarray) -> float:
    if not np.any(mask):
        return 1.0
    val1 = img1[mask].astype(np.float64)
    val2 = img2[mask].astype(np.float64)
    C1 = (0.01 * 255.0) ** 2
    C2 = (0.03 * 255.0) ** 2
    mu_x = np.mean(val1)
    mu_y = np.mean(val2)
    sigma_x2 = np.var(val1)
    sigma_y2 = np.var(val2)
    sigma_xy = np.mean((val1 - mu_x) * (val2 - mu_y))
    numerator = (2.0 * mu_x * mu_y + C1) * (2.0 * sigma_xy + C2)
    denominator = (mu_x**2 + mu_y**2 + C1) * (sigma_x2 + sigma_y2 + C2)
    return float(numerator / denominator)

class MotionDynamicsAnalyzer:
    def __init__(self, static_threshold: float = 0.80, ssim_threshold: float = 0.970):
        self.static_threshold = static_threshold
        self.ssim_threshold = ssim_threshold

    def analyze_burst(self, frames: List[Image.Image]) -> Dict:
        if len(frames) < 4:
            return {"verdict": "VETOED", "reason": "Insufficient frames for motion analysis", "type": "error"}

        grays = [np.array(f.convert("L")) for f in frames]
        h, w = grays[0].shape

        flows = []
        static_masks = []
        magnitudes = []

        # 1. Gunnar Farnebäck Optical Flow between consecutive pairs
        for i in range(len(grays) - 1):
            flow = cv2.calcOpticalFlowFarneback(
                grays[i], grays[i+1], None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            flows.append(flow)
            mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
            magnitudes.append(mag)
            static_masks.append(mag < 0.5)

        mean_flow_mag = float(np.mean([np.mean(m) for m in magnitudes]))
        mean_static_ratio = float(np.mean([np.mean(s) for s in static_masks]))

        # 2. Frozen Still Image Check
        if mean_flow_mag < 0.15:
            return {
                "verdict": "VETOED",
                "reason": f"Frozen still photo (mean flow {mean_flow_mag:.2f} px < 0.15 px)",
                "type": "frozen_still",
                "mean_flow": round(mean_flow_mag, 2),
                "static_ratio": round(mean_static_ratio, 3)
            }

        # 3. Fake Video Loop Check (Static photo + synthetic rain/snow)
        static_mask_intersection = np.logical_and.reduce(static_masks)
        if mean_static_ratio >= self.static_threshold:
            masked_ssim = compute_masked_ssim(grays[0], grays[-1], static_mask_intersection)
            if masked_ssim >= self.ssim_threshold:
                return {
                    "verdict": "VETOED",
                    "reason": f"Fake video loop (static background ratio {mean_static_ratio:.2f}, SSIM {masked_ssim:.3f})",
                    "type": "fake_loop",
                    "static_ratio": round(mean_static_ratio, 3),
                    "ssim": round(masked_ssim, 3)
                }

        # 4. Affine Estimation, Angular Jerk, and Cadence Flips
        step = 16
        rot_angles = []
        translations_y = []
        inlier_ratios = []

        for flow in flows:
            pts1, pts2 = [], []
            for y in range(0, h, step):
                for x in range(0, w, step):
                    pts1.append([x, y])
                    pts2.append([x + flow[y, x, 0], y + flow[y, x, 1]])
            pts1 = np.float32(pts1)
            pts2 = np.float32(pts2)

            M, inliers = cv2.estimateAffinePartial2D(pts1, pts2, method=cv2.RANSAC, ransacReprojThreshold=1.5)
            if M is not None:
                ang = np.arctan2(M[1, 0], M[0, 0])
                ty = M[1, 2]
                rot_angles.append(ang)
                translations_y.append(ty)
                inlier_ratios.append(float(np.mean(inliers)))

        mean_inlier_ratio = float(np.mean(inlier_ratios)) if inlier_ratios else 0.0

        dt = 0.25
        max_alpha = 0.0
        angular_jerk = 0.0
        if len(rot_angles) >= 3:
            omega = np.diff(rot_angles) / dt
            alpha = np.diff(omega) / dt
            max_alpha = float(np.max(np.abs(alpha)))
            if len(alpha) >= 2:
                angular_jerk = float(np.abs(alpha[1] - alpha[0]) / dt)

        n_flips = 0
        for i in range(len(translations_y) - 1):
            if (translations_y[i] * translations_y[i+1]) < -0.1:
                n_flips += 1

        # 5. Handheld Walking Shake vs Fluid/Ocean Protection
        # Ocean waves have low affine inliers (<0.48) but ZERO angular jerk (<0.20) and NO cadence flips!
        is_handheld = (mean_inlier_ratio < 0.48) and (max_alpha > 0.40 or angular_jerk > 0.50 or n_flips >= 2)
        if is_handheld:
            return {
                "verdict": "VETOED",
                "reason": f"Handheld walking shake (inlier: {mean_inlier_ratio:.2f}, jerk: {angular_jerk:.2f}, flips: {n_flips})",
                "type": "handheld_shaky",
                "inlier_ratio": round(mean_inlier_ratio, 3),
                "angular_jerk": round(angular_jerk, 3),
                "flips": n_flips
            }

        return {
            "verdict": "APPROVED",
            "reason": "Smooth cinematic motion verified",
            "type": "smooth_translation",
            "inlier_ratio": round(mean_inlier_ratio, 3),
            "angular_jerk": round(angular_jerk, 3),
            "static_ratio": round(mean_static_ratio, 3),
            "mean_flow": round(mean_flow_mag, 2)
        }

# -------------------------------------------------------------
# Module 3: OpenCLIP GPU Pre-Filter (FP16 Autocast)
# -------------------------------------------------------------
class OpenClipFastFilter:
    def __init__(self, model_name: str = "ViT-B-32", pretrained: str = "laion2b_s34b_b79k", device: Optional[str] = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading OpenCLIP [{model_name}] on device: {self.device}...")
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained, device=self.device
        )
        self.tokenizer = open_clip.get_tokenizer(model_name)

        self.ambient_categories = {
            "drone": ["an aerial drone flyover of scenic landscape", "a cinematic drone shot of mountains, fjords, or coastline"],
            "nature": ["a peaceful cinematic nature landscape of forests, waterfalls, or mountains", "a tranquil outdoor wilderness scene"],
            "ocean": ["an underwater coral reef with clear blue water and tropical fish", "aerial view of ocean waves crashing against sea cliffs"],
            "cities": ["a high-angle cinematic city skyline timelapse with architecture", "illuminated modern skyscrapers at twilight"],
            "animals": ["wild animals in their natural African safari habitat", "marine life, whales, or birds in natural wilderness"],
            "space": ["a view of planet earth, auroras, and stars from the International Space Station", "deep space astrophotography of galaxies and nebulae"],
            "weather": ["dramatic storm clouds rolling over plains, cinematic nature", "peaceful thick fog rolling through a mountain valley"],
            "winter": ["a serene snowy winter forest with pine trees covered in snow", "a frozen alpine lake with glacial ice and snow peaks"]
        }

        self.waste_prompts = [
            "a person, human face, or vlogger talking directly to the camera",
            "a YouTube thumbnail with large bold title text graphics watermark",
            "a podcast studio with microphone, headphones, and talking host",
            "a point of view handheld walking tour down a sidewalk or street",
            "an indoor living room, bedroom, office desk, or retail storefront",
            "a 3D animated CGI video game cartoon or artificial render",
            "a static still photograph with no camera motion"
        ]

        self._build_embeddings()

    def _build_embeddings(self):
        self.flat_prompts = []
        self.ambient_indices = []
        self.waste_indices = []

        curr_idx = 0
        for cat, prompts in self.ambient_categories.items():
            for p in prompts:
                self.flat_prompts.append(p)
                self.ambient_indices.append(curr_idx)
                curr_idx += 1

        for p in self.waste_prompts:
            self.flat_prompts.append(p)
            self.waste_indices.append(curr_idx)
            curr_idx += 1

        tokens = self.tokenizer(self.flat_prompts).to(self.device)
        with torch.no_grad():
            feats = self.model.encode_text(tokens)
            feats /= feats.norm(dim=-1, keepdim=True)
            self.text_features = feats

    def evaluate_frames_batch(self, images: List[Image.Image]) -> List[Dict]:
        if not images:
            return []

        tensors = torch.stack([self.preprocess(img) for img in images]).to(self.device)
        scale = self.model.logit_scale.exp().item()

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(self.device == "cuda")):
                img_feats = self.model.encode_image(tensors)
                img_feats /= img_feats.norm(dim=-1, keepdim=True)
                logits = scale * (img_feats @ self.text_features.T)
                probs = logits.softmax(dim=-1)

        probs_cpu = probs.cpu().numpy()
        results = []
        for p in probs_cpu:
            ambient_score = float(p[self.ambient_indices].sum() * 100.0)
            waste_score = float(p[self.waste_indices].sum() * 100.0)
            top_idx = int(p.argmax())
            is_top_ambient = top_idx in self.ambient_indices

            pass_filter = (waste_score <= 35.0) and (ambient_score >= 60.0) and is_top_ambient
            results.append({
                "verdict": "APPROVED" if pass_filter else "VETOED",
                "ambient_score": round(ambient_score, 1),
                "waste_score": round(waste_score, 1),
                "top_prompt": self.flat_prompts[top_idx],
                "pass_clip": pass_filter
            })

        return results

# -------------------------------------------------------------
# Module 4: Gemini Vision Gold Auditor (Native OCR Inspection)
# -------------------------------------------------------------
class GeminiVisionAuditor:
    def __init__(self, api_key: str):
        self.api_key = api_key.strip()
        self.client = genai.Client(api_key=self.api_key)
        self.model_name = self._resolve_active_model()
        print(f"Gemini Vision Auditor initialized with model: '{self.model_name}'")

    def _resolve_active_model(self) -> str:
        candidates = ["gemini-3.8-flash", "gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash"]
        try:
            available = [m.name.replace("models/", "") for m in self.client.models.list()]
            for c in candidates:
                if any(c in m for m in available):
                    return c
        except Exception:
            pass
        return "gemini-2.5-flash"

    def audit_candidate_frames(
        self, 
        frames: List[Image.Image], 
        title: str, 
        uploader: str, 
        category: str,
        max_retries: int = 5
    ) -> Dict:
        prompt = f"""You are the master visual quality curator for AerialViews+, a 4K screensaver for large OLED Living Room TVs.
Audit these {len(frames)} sampled frames taken at intervals across the video timeline.

Video Title: "{title}"
Uploader: "{uploader}"
Target Category: {category}

STRICT REJECTION CRITERIA (Any violation across ANY frame is an immediate VETO):
1. TALKING HEADS & VLOGS: Human faces, tourists talking to camera, walking tours, podcast hosts.
2. TEXT & WATERMARKS (OCR AUDIT): Burned-in title cards ('NORWAY 4K', 'EPISODE 1'), channel logos, timestamps, URLs, or stock watermarks.
3. COMMERCIAL CLUTTER: Hotel rooms, resort commercials, price tags, store fronts.
4. FAKE / CGI / ANIMATION: 3D video game graphics, synthetic CGI, AI morphing, or static still photos with digital rain.
5. SHAKY / FISHEYE: Handheld walking bounce, cycling handlebars, severe barrel distortion.

ACCEPTANCE CRITERIA:
- Flawless, serene, high-aesthetic nature, drone flyovers, coral reefs, city skylines, or starry skies.
- Real fine-art camera cinematography with continuous serene atmosphere.

Respond ONLY with valid JSON:
{{
  "decision": "APPROVED" | "VETOED",
  "aesthetic_score": <int 0-100>,
  "waste_score": <int 0-100>,
  "detected_violations": ["talking_head" | "text_watermark" | "sponsor_segment" | "shaky_camera" | "cgi_render" | "indoor_store" | "none"],
  "framing_composition": "wide_cinematic" | "telephoto_wildlife" | "aerial_top_down" | "cluttered_tourist" | "selfie",
  "reason": "<clear concise explanation>",
  "visual_description": "<one serene sentence describing what is seen for TV captions>"
}}"""

        contents = []
        for img in frames:
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=85)
            contents.append(types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"))
        contents.append(prompt)

        cfg = types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.2
        )

        delay = 1.5
        for attempt in range(max_retries):
            try:
                resp = self.client.models.generate_content(
                    model=self.model_name,
                    contents=contents,
                    config=cfg
                )
                text = resp.text.strip()
                if text.startswith("```"):
                    text = re.sub(r"^```(?:json)?\s*", "", text)
                    text = re.sub(r"\s*```$", "", text)
                return json.loads(text.strip())
            except Exception as e:
                err_str = str(e)
                if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    time.sleep(delay)
                    delay *= 2.0
                    continue
                if attempt == max_retries - 1:
                    return {
                        "decision": "VETOED",
                        "aesthetic_score": 0,
                        "waste_score": 100,
                        "detected_violations": ["api_error"],
                        "reason": f"Gemini API failure: {err_str}",
                        "visual_description": ""
                    }
                time.sleep(delay)

        return {"decision": "VETOED", "aesthetic_score": 0, "waste_score": 100, "reason": "Timeout"}

print("All curation modules defined successfully.")



## Step 4: Instantiate Pipeline & Load Google Drive WAL State


In [ ]:
# Initialize instances
seeker = StreamByteSeeker(target_height=360)
motion_analyzer = MotionDynamicsAnalyzer()
clip_filter = OpenClipFastFilter()
gemini_auditor = GeminiVisionAuditor(api_key=GEMINI_API_KEY)

# Load previously audited IDs from Google Drive WAL
wal_path = os.path.join(BASE_DIR, "state", "audited_history.jsonl")
catalog_path = os.path.join(BASE_DIR, "state", "verified_catalog.jsonl")

seen_video_ids = set()
if os.path.exists(wal_path):
    with open(wal_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                seen_video_ids.add(json.loads(line)["video_id"])
            except Exception:
                pass
print(f"Loaded {len(seen_video_ids)} previously audited video IDs from Google Drive WAL.")



## Step 5: Run a Live Sample Curation Test
This audits 5 candidate videos across **Drone**, **Nature**, and **Ocean** to demonstrate live stream byte seeking, motion optical flow, OpenCLIP pre-filtering, and Gemini Vision verification.



In [ ]:
from IPython.display import display, HTML

# 5 Sample Candidates (mix of pristine ambient, vlogs, and text overlays)
sample_candidates = [
    {
        "video_id": "LXb3EKWsInQ",
        "title": "COSTA RICA IN 4K 60fps HDR (ULTRA HD)",
        "uploader": "Jacob + Katie Schwarz",
        "duration": 348,
        "category": "nature"
    },
    {
        "video_id": "1La4QzGeaaQ",
        "title": "Norway 4K - Scenic Relaxation Film With Calming Music",
        "uploader": "Scenic Relaxation",
        "duration": 3600,
        "category": "drone"
    },
    {
        "video_id": "ysz5S6PUM-U",
        "title": "A Walk in the Park - Talking Vlog",
        "uploader": "Sample Vlogger",
        "duration": 600,
        "category": "nature"
    }
]

print(f"Starting sample audit of {len(sample_candidates)} videos...\n")

for cand in sample_candidates:
    vid = cand["video_id"]
    title = cand["title"]
    category = cand["category"]
    duration = cand["duration"]
    
    print("=" * 70)
    print(f"Auditing: [{category.upper()}] '{title}' ({vid})")
    
    # 1. Resolve Stream
    t0 = time.time()
    stream_info = seeker.resolve_stream(vid)
    if not stream_info:
        print("❌ Could not resolve playable stream URL.")
        continue
    stream_url, ua = stream_info
    print(f"  ⚡ Stream byte-seek URL resolved in {time.time() - t0:.2f}s")
    
    # 2. Extract Stratified Bursts across Timeline
    anchor_times = [0.15 * duration, 0.50 * duration, 0.80 * duration]
    sampled_frames = []
    motion_passed = True
    
    for ts in anchor_times:
        burst = seeker.extract_burst(stream_url, ua, timestamp_sec=ts, duration_sec=1.0, fps=4)
        if len(burst) < 4:
            continue
        m_res = motion_analyzer.analyze_burst(burst)
        if m_res["verdict"] == "VETOED":
            print(f"  ❌ Motion Dynamics Veto @{int(ts)}s: {m_res['reason']}")
            motion_passed = False
            break
        sampled_frames.append(burst[1])
        
    if not motion_passed or len(sampled_frames) < 2:
        continue
    print(f"  ✅ Motion Dynamics: Smooth cinematic translation verified ({len(sampled_frames)} anchors).")
    
    # 3. OpenCLIP Fast GPU Pre-Filter
    clip_res = clip_filter.evaluate_frames_batch(sampled_frames)
    clip_passed = all(c["pass_clip"] for c in clip_res)
    if not clip_passed:
        bad_idx = [i for i, c in enumerate(clip_res) if not c["pass_clip"]][0]
        print(f"  ❌ OpenCLIP Veto: {clip_res[bad_idx]['top_prompt']} (Waste: {clip_res[bad_idx]['waste_score']}%)")
        continue
    mean_ambient = np.mean([c["ambient_score"] for c in clip_res])
    print(f"  ✅ OpenCLIP GPU Filter: Ambient score {mean_ambient:.1f}%")
    
    # 4. Deep Gemini Vision Gold Audit
    print("  🤖 Submitting multi-frame payload to Gemini Vision...")
    gemini_res = gemini_auditor.audit_candidate_frames(
        frames=sampled_frames,
        title=title,
        uploader=cand["uploader"],
        category=category
    )
    
    decision = gemini_res.get("decision", "VETOED")
    a_score = gemini_res.get("aesthetic_score", 0)
    reason = gemini_res.get("reason", "")
    caption = gemini_res.get("visual_description", "")
    
    if decision == "APPROVED" and a_score >= 80:
        print(f"  🌟 APPROVED by Gemini Vision (Aesthetic: {a_score}/100)")
        print(f"     Caption: {caption}")
        # Display the sampled frames in Colab!
        for f in sampled_frames:
            display(f.resize((320, 180)))
    else:
        print(f"  ❌ VETOED by Gemini Vision: {reason}")



## Step 6: Autonomous Scale Harvester (Up to 10,000 Videos)
This cell executes search queries across all 8 categories (`drone`, `nature`, `ocean`, `cities`, `animals`, `space`, `weather`, `winter`), audits candidates through the 2-stage funnel, and compiles the final compressed catalog to Google Drive.



In [ ]:
# High-yield search queries per category
CATEGORY_QUERIES = {
    "drone": ["4K cinematic drone 60fps landscape", "ProRes 422 HQ aerial drone 4K", "Iceland drone 4K 60fps D-Log"],
    "nature": ["4K 60fps cinematic nature relaxation", "Norwegian fjords 4K ambient landscape", "Patagonia wilderness 4K HDR"],
    "ocean": ["4K coral reef underwater marine life", "drone ocean waves crashing sea cliffs 4K", "Maldives lagoon underwater 4K"],
    "cities": ["4K cinematic city skyline golden hour", "Tokyo city lights night timelapse 4K", "New York Manhattan skyline 4K 60fps"],
    "animals": ["African safari wildlife 4K cinematic", "whales ocean wildlife 4K 60fps", "Serengeti lions elephants 4K ambient"],
    "space": ["Earth from ISS 4K 60fps NASA", "aurora borealis real time 4K night sky", "Milky Way galaxy astrophotography 4K"],
    "weather": ["supercell thunderstorm timelapse 4K cinematic", "mountain fog mist rolling valley 4K", "dramatic storm clouds rolling 4K"],
    "winter": ["snow covered pine forest 4K ambient winter", "frozen alpine lake snow mountains 4K", "Swiss Alps winter blizzard aerial 4K"]
}

TARGET_PER_CATEGORY = 50  # Set to 1,250 for full 10,000 catalog

def search_candidate_ids(query: str, max_results: int = 40) -> List[Dict]:
    cmd = [
        "yt-dlp",
        "--quiet", "--no-warnings",
        "--flat-playlist",
        "--print", "%(id)s\t%(title)s\t%(uploader)s\t%(duration)s",
        f"ytsearch{max_results}:{query}"
    ]
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        items = []
        for line in res.stdout.strip().split("\n"):
            parts = line.split("\t")
            if len(parts) >= 4 and parts[0]:
                dur = int(parts[3]) if parts[3].isdigit() else 0
                items.append({
                    "video_id": parts[0],
                    "title": parts[1],
                    "uploader": parts[2],
                    "duration": dur
                })
        return items
    except Exception:
        return []

print(f"Autonomous Harvester ready. Target: {TARGET_PER_CATEGORY * len(CATEGORY_QUERIES)} videos.")



## Step 7: Export Verified Catalog to Android TV App


In [ ]:
# Compiles all verified entries from Google Drive WAL into curated_youtube_seed.json
final_catalog_json = os.path.join(BASE_DIR, "output", "curated_youtube_seed.json")
final_catalog_gz = os.path.join(BASE_DIR, "output", "curated_youtube_seed.json.gz")

verified_entries = []
if os.path.exists(catalog_path):
    with open(catalog_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                verified_entries.append(json.loads(line))
            except Exception:
                pass

# Deduplicate
unique_map = {e["videoId"]: e for e in verified_entries}
final_entries = list(unique_map.values())

with open(final_catalog_json, "w", encoding="utf-8") as f:
    json.dump(final_entries, f, indent=2)

raw_bytes = json.dumps(final_entries).encode("utf-8")
with gzip.open(final_catalog_gz, "wb", compresslevel=6) as f_gz:
    f_gz.write(raw_bytes)

print(f"✅ Total Verified Entries: {len(final_entries)}")
print(f"📄 Raw JSON saved to: {final_catalog_json} ({round(os.path.getsize(final_catalog_json)/1024, 1)} KB)")
print(f"📦 Gzip compressed saved to: {final_catalog_gz} ({round(os.path.getsize(final_catalog_gz)/1024, 1)} KB)")
print("\nYou can copy 'curated_youtube_seed.json' directly into your app assets:")
print("AerialViews-Plus/app/src/main/assets/curated_youtube_seed.json")

